# Simulation Analysis

This notebook analyzes the outputs of the Monte Carlo tournament simulation used in the **World Cup 2026 Forecast** project.

The objective is to evaluate the tournament-level behavior of the forecasting engine, inspect team advancement probabilities, understand champion distributions, and assess simulation stability.

This notebook focuses on:

- forecast output inspection
- team progression probabilities
- champion distribution analysis
- upset and uncertainty patterns
- simulation stability and convergence
- interpretation of tournament-level results

## Project paths setup

Resolve project root and load simulation artifacts generated by the tournament simulation engine.

In [ ]:
from pathlib import Path

# Detect project root
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Simulation outputs directory
SIM_OUTPUT = PROJECT_ROOT / "data" / "outputs" / "simulation"

team_probs = SIM_OUTPUT / "team_probabilities.csv"
champion_dist = SIM_OUTPUT / "champion_distribution.csv"
summary_metadata_path = SIM_OUTPUT / "summary_metadata.json"
match_logs_path = SIM_OUTPUT / "match_logs.parquet"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

team_probabilities = pd.read_csv(team_probs)
champion_distribution = pd.read_csv(champion_dist)

summary_metadata = None
if summary_metadata_path.exists():
    with open(summary_metadata_path) as f:
        summary_metadata = json.load(f)

match_logs_df = None
if match_logs_path.exists():
    match_logs_df = pd.read_parquet(match_logs_path)

In [ ]:
top_10_champion = champion_distribution.sort_values(
    "champion_prob",
    ascending=False
).head(10)

plt.figure(figsize=(10,6))
plt.barh(top_10_champion["team"], top_10_champion["champion_prob"])

plt.xlabel("Champion probability")
plt.ylabel("Team")
plt.title("Top 10 Teams by Championship Probability")

plt.gca().invert_yaxis()
plt.tight_layout()

output_dir = PROJECT_ROOT / "docs" / "images"
output_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(output_dir / "champion_probabilities.png", dpi=300)

plt.show()

In [ ]:
top_10_distribution = (
    champion_distribution
    .sort_values("champion_prob", ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 6))
plt.barh(top_10_distribution["team"], top_10_distribution["champion_prob"])

plt.xlabel("Champion probability")
plt.ylabel("Team")
plt.title("Champion Distribution Across Simulations")

plt.gca().invert_yaxis()
plt.tight_layout()

output_dir = PROJECT_ROOT / "docs" / "images"
output_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(output_dir / "champion_distribution.png", dpi=300)

plt.show()

## Project paths

This notebook expects simulation outputs exported by the reporting layer, including:

- `team_probabilities.csv`
- `champion_distribution.csv`
- `match_logs.parquet`
- `summary_metadata.json`

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
SIM_OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs" / "simulation"

TEAM_PROB_PATH = SIM_OUTPUT_DIR / "team_probabilities.csv"
CHAMPION_DIST_PATH = SIM_OUTPUT_DIR / "champion_distribution.csv"
MATCH_LOGS_PATH = SIM_OUTPUT_DIR / "match_logs.parquet"
SUMMARY_METADATA_PATH = SIM_OUTPUT_DIR / "summary_metadata.json"

PROJECT_ROOT, SIM_OUTPUT_DIR

In [ ]:
required_paths = [
    TEAM_PROB_PATH,
    CHAMPION_DIST_PATH,
    MATCH_LOGS_PATH,
    SUMMARY_METADATA_PATH,
]

missing_paths = [str(path) for path in required_paths if not path.exists()]

if missing_paths:
    raise FileNotFoundError(
        "Missing simulation artifacts:\n- " + "\n- ".join(missing_paths)
    )

print("All required simulation artifacts found.")

## Load simulation artifacts

In [ ]:
team_prob_df = pd.read_csv(TEAM_PROB_PATH)
champion_dist_df = pd.read_csv(CHAMPION_DIST_PATH)
match_logs_df = pd.read_parquet(MATCH_LOGS_PATH)

with open(SUMMARY_METADATA_PATH, "r", encoding="utf-8") as f:
    summary_metadata = json.load(f)

print("team_prob_df:", team_prob_df.shape)
print("champion_dist_df:", champion_dist_df.shape)
print("match_logs_df:", match_logs_df.shape)
summary_metadata

## Metadata summary

In [ ]:
metadata_df = pd.DataFrame([summary_metadata])
metadata_df

## Team probability table overview

In [ ]:
team_prob_df.head(10)

In [ ]:
team_prob_df.columns.tolist()

## Sort teams by championship probability

In [ ]:
team_prob_sorted = team_prob_df.sort_values("champion_prob", ascending=False).reset_index(drop=True)
team_prob_sorted.head(20)

## Top teams by championship probability

In [ ]:
top_n = min(20, len(team_prob_sorted))
top_champ = team_prob_sorted.head(top_n).copy()

plt.figure(figsize=(10, 6))
plt.barh(top_champ["team"][::-1], top_champ["champion_prob"][::-1])
plt.xlabel("Champion probability")
plt.ylabel("Team")
plt.title("Top Teams by Champion Probability")
plt.tight_layout()
plt.show()

## Progression probability columns

In [ ]:
probability_cols = [col for col in team_prob_df.columns if col.endswith("_prob")]
probability_cols

## Team progression table

In [ ]:
team_prob_sorted.head(15)

## Advancement ladder visualization for top contenders

In [ ]:
stage_order = [
    col for col in [
        "advance_from_group_prob",
        "round_of_32_prob",
        "round_of_16_prob",
        "quarterfinal_prob",
        "semifinal_prob",
        "final_prob",
        "champion_prob",
    ]
    if col in team_prob_sorted.columns
]

top_k = min(8, len(team_prob_sorted))
ladder_df = (
    team_prob_sorted.loc[:top_k-1, ["team"] + stage_order]
    .set_index("team")
    .T
)

ladder_df

In [ ]:
if len(stage_order) > 0:
    plt.figure(figsize=(11, 6))
    for team in ladder_df.columns:
        plt.plot(ladder_df.index, ladder_df[team], marker="o", label=team)

    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Probability")
    plt.title("Advancement Probability Ladder for Top Contenders")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("No stage probability columns found.")

## Champion distribution

In [ ]:
champion_dist_sorted = champion_dist_df.sort_values("champion_prob", ascending=False).reset_index(drop=True)
champion_dist_sorted.head(20)

In [ ]:
top_n = min(20, len(champion_dist_sorted))
top_champion_dist = champion_dist_sorted.head(top_n)

plt.figure(figsize=(10, 6))
plt.barh(top_champion_dist["team"][::-1], top_champion_dist["champion_prob"][::-1])
plt.xlabel("Champion probability")
plt.ylabel("Team")
plt.title("Champion Distribution")
plt.tight_layout()
plt.show()

## Forecast concentration

In [ ]:
top_5_mass = champion_dist_sorted["champion_prob"].head(5).sum()
top_10_mass = champion_dist_sorted["champion_prob"].head(10).sum()
entropy = -(champion_dist_sorted["champion_prob"] * np.log(np.clip(champion_dist_sorted["champion_prob"], 1e-12, 1))).sum()

pd.DataFrame([{
    "top_5_champion_probability_mass": top_5_mass,
    "top_10_champion_probability_mass": top_10_mass,
    "champion_distribution_entropy": entropy,
}]).round(4)

## Match log overview

In [ ]:
match_logs_df.head()

In [ ]:
match_logs_df.columns.tolist()

## Stage distribution in the simulation logs

In [ ]:
if "stage" in match_logs_df.columns:
    stage_counts = match_logs_df["stage"].value_counts().sort_index()
    stage_counts.to_frame("num_matches")
else:
    print("No 'stage' column found in match logs.")

## Outcome distribution in simulated matches

In [ ]:
if "outcome" in match_logs_df.columns:
    outcome_counts = match_logs_df["outcome"].value_counts(normalize=True).sort_index()
    outcome_counts.plot(kind="bar", figsize=(7, 4))
    plt.title("Simulated Match Outcome Distribution")
    plt.xlabel("Outcome")
    plt.ylabel("Share of simulated matches")
    plt.tight_layout()
    plt.show()
else:
    print("No 'outcome' column found in match logs.")

## Upset analysis

We define a simple upset proxy using pre-match win probabilities:

- if Team A wins with a low Team A win probability
- if Team B wins with a low Team B win probability inferred from the logged probabilities

This is not a perfect upset definition, but it provides a useful high-level signal.

In [ ]:
upset_df = match_logs_df.copy()

required_upset_cols = {"team_a", "team_b", "team_a_win_prob", "team_b_win_prob", "winner"}
if required_upset_cols.issubset(upset_df.columns):
    upset_df["winner_pre_match_prob"] = np.where(
        upset_df["winner"] == upset_df["team_a"],
        upset_df["team_a_win_prob"],
        np.where(
            upset_df["winner"] == upset_df["team_b"],
            upset_df["team_b_win_prob"],
            np.nan,
        ),
    )

    upset_df["is_upset_20"] = (upset_df["winner_pre_match_prob"] < 0.20).astype(int)
    upset_df["is_upset_30"] = (upset_df["winner_pre_match_prob"] < 0.30).astype(int)

    pd.DataFrame([{
        "upset_rate_below_20pct": upset_df["is_upset_20"].mean(),
        "upset_rate_below_30pct": upset_df["is_upset_30"].mean(),
    }]).round(4)
else:
    print("Required probability columns for upset analysis are not available.")

In [ ]:
if "winner_pre_match_prob" in upset_df.columns:
    upset_df["winner_pre_match_prob"].hist(bins=30, figsize=(8, 4))
    plt.title("Distribution of Winner Pre-Match Probability")
    plt.xlabel("Winner pre-match probability")
    plt.ylabel("Number of simulated matches")
    plt.tight_layout()
    plt.show()

## Most surprising simulated matches

In [ ]:
if "winner_pre_match_prob" in upset_df.columns:
    surprising_matches = (
        upset_df.sort_values("winner_pre_match_prob", ascending=True)
        .head(20)
    )
    surprising_matches
else:
    print("Upset proxy not available.")

## Team-level uncertainty profile

In [ ]:
if set(["final_prob", "champion_prob"]).issubset(team_prob_sorted.columns):
    uncertainty_profile = team_prob_sorted[["team", "final_prob", "champion_prob"]].copy()
    uncertainty_profile["conversion_final_to_title"] = np.where(
        uncertainty_profile["final_prob"] > 0,
        uncertainty_profile["champion_prob"] / uncertainty_profile["final_prob"],
        np.nan,
    )
    uncertainty_profile.sort_values("final_prob", ascending=False).head(20)
else:
    print("final_prob and champion_prob are required for this section.")

## Simulation stability and convergence

To study stability, we recompute champion probability estimates using increasing prefixes of simulation runs.

This requires a `simulation_id` column in the match logs and one final-stage match per tournament run.

In [ ]:
if {"simulation_id", "stage", "winner"}.issubset(match_logs_df.columns):
    final_stage_candidates = [stage for stage in match_logs_df["stage"].dropna().unique() if "final" in str(stage).lower()]
    final_stage_candidates
else:
    print("Required columns for convergence analysis are not available.")

In [ ]:
convergence_df = None

if {"simulation_id", "stage", "winner"}.issubset(match_logs_df.columns):
    final_stage_candidates = [stage for stage in match_logs_df["stage"].dropna().unique() if "final" in str(stage).lower()]

    if len(final_stage_candidates) > 0:
        final_stage_name = sorted(final_stage_candidates, key=lambda x: str(x))[-1]

        final_matches = (
            match_logs_df[match_logs_df["stage"] == final_stage_name]
            .sort_values("simulation_id")
            .copy()
        )

        champions_by_run = final_matches[["simulation_id", "winner"]].dropna().rename(columns={"winner": "champion"})

        checkpoints = []
        total_runs = len(champions_by_run)

        if total_runs > 0:
            for n in [100, 500, 1000, 5000, 10000, total_runs]:
                if n <= total_runs:
                    tmp = (
                        champions_by_run.head(n)["champion"]
                        .value_counts(normalize=True)
                        .rename_axis("team")
                        .reset_index(name="champion_prob_prefix")
                    )
                    tmp["num_simulations"] = n
                    checkpoints.append(tmp)

            if checkpoints:
                convergence_df = pd.concat(checkpoints, ignore_index=True)

convergence_df.head() if convergence_df is not None else print("Convergence analysis unavailable for current logs.")

In [ ]:
if convergence_df is not None:
    top_teams = champion_dist_sorted.head(min(5, len(champion_dist_sorted)))["team"].tolist()
    plot_df = convergence_df[convergence_df["team"].isin(top_teams)].copy()

    plt.figure(figsize=(9, 5))
    for team in top_teams:
        team_df = plot_df[plot_df["team"] == team].sort_values("num_simulations")
        if len(team_df) > 0:
            plt.plot(team_df["num_simulations"], team_df["champion_prob_prefix"], marker="o", label=team)

    plt.xlabel("Number of simulations")
    plt.ylabel("Champion probability estimate")
    plt.title("Convergence of Champion Probability Estimates")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("Convergence plot unavailable.")

## Key findings

This section should be updated after running the notebook on the latest forecast artifacts.

Suggested interpretation points:

- identify the main title favorites
- assess how concentrated or open the tournament looks
- inspect whether advancement ladders are smooth and coherent
- evaluate whether upset rates look realistic
- confirm that champion probabilities stabilize as the number of simulations increases

## Simulation results overview

The Monte Carlo simulation provides a probabilistic view of the tournament landscape by repeatedly sampling match outcomes using the trained match prediction model.

Across the simulated tournaments, the forecast identifies a small group of clear contenders while still maintaining substantial uncertainty in the overall champion distribution.

---

## Tournament favorites

The simulation results indicate that **Spain and Argentina emerge as the strongest title contenders**, with championship probabilities of approximately **23% and 21% respectively**.

A second tier of competitive teams follows, including **Colombia, France, and England**, each with championship probabilities between roughly **6% and 9%**.

Beyond this group, several traditionally strong national teams such as **Portugal, Brazil, and the Netherlands** maintain moderate title chances in the **4–5% range**, reflecting both their underlying strength and the stochastic nature of tournament play.

---

## Tournament progression dynamics

The progression probabilities reveal important differences between teams that are likely to advance deep into the tournament and those whose chances decline rapidly in later knockout rounds.

For example:

* **Spain and Argentina** maintain strong progression probabilities throughout all stages, with semifinal probabilities above **40%**.
* **Colombia** shows particularly strong quarterfinal and semifinal probabilities, suggesting a favorable combination of team strength and bracket dynamics.
* Several teams exhibit high probabilities of reaching the **Round of 16** but significantly lower semifinal or final probabilities, indicating that their overall tournament winning chances depend on favorable knockout matchups.

This progression analysis helps distinguish **true title contenders** from teams that are likely to perform well in the early knockout rounds but struggle against elite opponents.

---

## Forecast concentration and uncertainty

The concentration analysis highlights the degree of uncertainty in the tournament forecast:

* The **top 5 teams account for approximately 66.6%** of total championship probability.
* The **top 10 teams account for roughly 87.1%** of the probability mass.

This indicates that while the forecast identifies a clear group of leading contenders, a non-trivial portion of championship probability remains distributed across a broader set of teams.

The **champion distribution entropy (~2.48)** further reflects this uncertainty, suggesting that the tournament outcome remains meaningfully unpredictable despite the presence of strong favorites.

---

## Implications for tournament forecasting

These results illustrate the value of combining **match-level probabilistic models with Monte Carlo tournament simulation**.

While the match model captures relative team strength and contextual features, the tournament simulation translates those probabilities into **stage-level advancement likelihoods and championship forecasts**, enabling a more realistic representation of knockout competition dynamics.

This approach allows analysts to quantify:

* championship probabilities,
* stage advancement likelihoods,
* and the overall uncertainty of the tournament outcome.

---

## Limitations and future improvements

Several extensions could further improve the realism and predictive performance of the simulation framework:

1. **Improved match prediction models**, including gradient boosting methods such as LightGBM or XGBoost.
2. **More detailed goal-based models**, which could capture score distributions rather than only match outcomes.
3. **More advanced probability calibration**, ensuring that predicted probabilities translate accurately into simulated tournament outcomes.
4. **Scenario analysis**, exploring how alternative group-stage draws or team strength updates affect the forecast.

---

## Next steps

Planned follow-up work:

1. Re-run the notebook once the final 2026 qualified teams and official groups are known.
2. Compare forecast outputs across alternative match prediction models.
3. Assess whether calibration improvements at match level translate into better tournament-level probabilities.
4. Add richer visualizations for bracket progression and team-to-team comparison.
5. Extend the simulation engine toward goal-based scoreline models.